In [26]:
import torch
import torch.nn as nn

In [27]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r, alpha):
        super().__init__()
        self.base_layer = base_layer
        self.A = nn.Parameter(torch.randn(r, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, r))
        self.scale = alpha / r
        for param in base_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        self.BA = (self.B @ self.A) 
        frozen = self.base_layer(x)
        LoRA = (x @ self.BA.T) * self.scale

        return frozen + LoRA

In [28]:
base_model = nn.Linear(128, 22)

print(sum(p.numel() for p in base_model.parameters()if p.requires_grad)) 

2838


In [29]:
model = LoRALinear(nn.Linear(128, 22), 2, 10)

dummy = torch.randn(1, 256, 128)

out = model(dummy)
out2 = model.base_layer(dummy)

print(torch.allclose(out, out2))

print(sum(p.numel() for p in model.parameters()if p.requires_grad)) 

True
300


In [30]:
def compute_attention(Q, K, d_model):
    scores = Q @ K.transpose(-2, -1) / d_model**0.5

    return scores

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, r=2, alpha=10):
        super().__init__()
        self.d_model = d_model
        self.Wq = LoRALinear(nn.Linear(d_model, d_model), r=r, alpha=alpha)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = LoRALinear(nn.Linear(d_model, d_model), r=r, alpha=alpha)
        self.Wo = nn.Linear(d_model, d_model)


    def forward(self, x):
        scores = compute_attention(self.Wq(x), self.Wk(x), self.d_model)

        weights = torch.softmax(scores, dim=-1)

        outputs = weights @ self.Wv(x)

        return self.Wo(outputs)


In [31]:
dummy_in = torch.randn(2, 5, 16)

head_lora = SingleHeadAttention(16)
out_lora = head_lora(dummy_in)

Q = head_lora.Wq.base_layer(dummy_in)
K = head_lora.Wk(dummy_in)
V = head_lora.Wv.base_layer(dummy_in)

scores = compute_attention(Q, K, d_model=head_lora.d_model)

weights = torch.softmax(scores, dim=-1)

outputs = weights @ V

manual_out = head_lora.Wo(outputs)

print(torch.allclose(manual_out, out_lora))

True
